# Generate VHH sequences + random physchem data.

The purpose of this notebook is to generate fake VHH sequences and data for prototyping selection and prediction models.

In [3]:
import random
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

from tqdm.notebook import tqdm

Protein and data variables.

In [4]:
readout = 'vhh_sequences'

In [5]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

### Generate fake VHHs.

In [6]:
n_sequences = 30   # number of fake VHH sequences
seq_len = 120      # VHH length

In [7]:
# automatically annote sequences by region (CDRs, etc.)
framework_positions = list(range(0, 30)) + list(range(45, 60)) + list(range(67, 95)) + list(range(115, 120))
cdr_positions = list(set(range(seq_len)) - set(framework_positions))

In [8]:
aa_list = list("ACDEFGHIKLMNPQRSTVWY")

In [9]:
sequences = []
for _ in range(n_sequences):
    seq = []
    for i in range(seq_len):
        if i in framework_positions:
            seq.append(random.choice(['A','L','V','S','T']))  # residues most likely to be conserved
        else:
            seq.append(random.choice(aa_list))                # variable CDR
    sequences.append(''.join(seq))

In [11]:
data = []

for seq in sequences:
    # Base random properties (reference/no-heparin)
    Tm_no_heparin = round(random.uniform(55, 75), 1)
    solubility_no_heparin = round(random.uniform(0.5, 1.0), 2)
    expression_no_heparin = round(random.uniform(0.2, 1.0), 2)

    # Heparin condition: add a random shift to simulate effect
    # Positive or negative effect randomly
    Tm_heparin = round(Tm_no_heparin + random.uniform(-5, 5), 1)
    solubility_heparin = round(min(max(solubility_no_heparin + random.uniform(-0.2, 0.2), 0), 1), 2)
    expression_heparin = round(min(max(expression_no_heparin + random.uniform(-0.2, 0.2), 0), 1), 2)

    data.append({
        'sequence': seq,
        'Tm_no_heparin': Tm_no_heparin,
        'Tm_heparin': Tm_heparin,
        'solubility_no_heparin': solubility_no_heparin,
        'solubility_heparin': solubility_heparin,
        'expression_no_heparin': expression_no_heparin,
        'expression_heparin': expression_heparin
    })

In [12]:
df = pd.DataFrame(data)

In [14]:
df.to_csv(f"{DATA}/fake_vhh_multi_conditions.csv", index=False)